# ML-07: Baseline Action Score and Top-10 Review

## 1. Lane and two signal checks

**Confirmed lane:** CTR / Engagement Opportunity Scoring. The first version focuses on search CTR; engagement needs separate availability-aware work.

**Rule:** among pages with 500+ first-half impressions, all 15 days of search availability and a valid mean position of 1–20, compare observed CTR with the median for the position band in development clients. Rank positive gaps by `gap_in_percentage_points / 100 × first_half_impressions`. A content specialist reviews search intent and the snippet before deciding whether to edit.

Signal tables use only first-half measurements from training clients (hash folds 2–4). Neither later CTR nor final-test outcomes are used. The rule is frozen before capstone modeling.

In [1]:
from pathlib import Path
import sys
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p/'work/ctr_study.py').is_file())
sys.path.insert(0,str(ROOT/'work'))
import ctr_study as study
import numpy as np
import pandas as pd
from IPython.display import display
import hashlib
frame=study.make_frame('march')
frame['fold']=frame.client_hash_id.map(lambda x:int(hashlib.sha256(('ctr-v1:'+x).encode()).hexdigest()[:8],16)%5)
reference=frame.loc[frame.fold>=2].copy()
position_table=reference.groupby('position_band',observed=True).agg(n=('past_ctr','size'),median_ctr_pp=('past_ctr','median'))
reference['volume_band']=pd.cut(reference.past_impressions,[499,1000,3000,10000,np.inf],labels=['500-1000','1001-3000','3001-10000','10001+'])
volume_table=reference.groupby('volume_band',observed=True).agg(n=('past_ctr','size'),median_ctr_pp=('past_ctr','median'),median_zero_click_day_fraction=('zero_click_fraction','median'))
display(position_table.round(5)); display(volume_table.round(5))


,n,median_ctr_pp
position_band,,
1-3,543,0.28439
>3-10,4844,0.39120
>10-20,2913,0.24215


,n,median_ctr_pp,median_zero_click_day_fraction
volume_band,,,
500-1000,3886,0.35800,0.86667
1001-3000,3182,0.28524,0.73333
3001-10000,1104,0.32610,0.40000
10001+,128,0.29664,0.10000


**CTR-vs-position: MIXED.** This is the observable signal behind CTR-fix logic, not a claim about the exact FlyRank production flag. Median CTR is 0.28439% at positions 1–3, 0.39120% at >3–10, and 0.24215% at >10–20. The first two bands do not follow a monotonic curve. Use empirical band references, not a fabricated universal position curve; page/client/query mix may explain the pattern.

**Volume-vs-zero-click days: CONFIRMED.** Median zero-click-day fractions fall from 0.86667 to 0.73333, 0.40000 and 0.10000 across increasing impression bands. Volume matters for measurement stability and reviewer scale. This does not establish that high-volume pages are more fixable; only 128 reference pages lie in the highest volume bucket.

## 2. One score, one reason code, one action

The score has units of a **directional click gap over the first-half exposure**, not promised recoverable clicks. Raw identifiers remain pseudonyms; the CSV is local and ignored by Git.

In [2]:
queue=study.baseline_queue(frame,reference)
cols=['rank','client_hash_id','content_hash_id','score','gap_pp','past_impressions','past_ctr','reference_ctr','mean_position','reason_code','action']
path=study.OUT/'baseline_action_score.csv'
queue[cols].to_csv(path,index=False)
assert path.is_file() and len(pd.read_csv(path))==len(queue)
assert queue.score.gt(0).all() and queue.score.is_monotonic_decreasing
print(f'Wrote {len(queue):,} ranked candidates to work/outputs/baseline_action_score.csv')
print('Reason:',queue.reason_code.unique().tolist())
print('Action:',queue.action.unique().tolist())


Wrote 18,445 ranked candidates to work/outputs/baseline_action_score.csv
Reason: ['BELOW_POSITION_REFERENCE']
Action: ['Review search intent and snippet']


## 3. Top ten: evidence review

These are **numeric evidence reviews**, not independent inspections of the real pages: titles, URLs and queries are unavailable. Every action is a review request, not an assertion that a specific title is defective. The caveats below identify a distinct way each recommendation can be wrong.

In [3]:
caveats=[
 'High impression volume can amplify a small gap; the query mix may differ from reference pages.',
 'A navigational or zero-click query mix could explain low CTR without a snippet defect.',
 'The broad position band may mix very different search-result layouts.',
 'Daily average position hides query-level rank changes and exposure concentration.',
 'Reference-client medians may not transfer to this client or content niche.',
 'Low CTR may be appropriate for informational queries answered on the results page.',
 'A few heavily exposed queries can dominate the aggregate; query-level checks are needed.',
 'The 15-day window may reflect a temporary event rather than a persistent problem.',
 'An already appropriate title could be harmed by an unnecessary rewrite.',
 'Client measurement coverage does not guarantee comparable device/country intent mix.'
]
review=queue.head(10).copy()
review['why']=review.apply(lambda r:f"{r.past_impressions:,.0f} impressions; CTR {r.past_ctr:.3f}% vs band reference {r.reference_ctr:.3f}%; gap score {r.score:.1f}.",axis=1)
review['what_would_make_it_wrong']=caveats
review['confidence']='Directional evidence only; human inspection required'
display(review[['rank','content_hash_id','action','why','what_would_make_it_wrong']])
review[['rank','content_hash_id','action','why','what_would_make_it_wrong']].to_csv(study.OUT/'top10_review.csv',index=False)
study.save_json('baseline_metrics.json',{'candidate_pages':len(queue),'top10_clients':int(review.client_hash_id.nunique()),
    'feature_pages':len(frame),'reference_pages':len(reference),'signal_verdicts':{'position':'MIXED','volume':'CONFIRMED'},
    'score_formula':'max(position_band_reference_ctr - past_ctr, 0) / 100 * past_impressions',
    'independent_actionability_precision_at_10':None})


,rank,content_hash_id,action,why,what_would_make_it_wrong
0,1,content_7c6373141eae744a,Review search intent and snippet,"86,860 impressions; CTR 0.059% vs band referen...",High impression volume can amplify a small gap...
1,2,content_8e1334d6356668e3,Review search intent and snippet,"58,553 impressions; CTR 0.002% vs band referen...",A navigational or zero-click query mix could e...
2,3,content_65c75874a23fca87,Review search intent and snippet,"55,680 impressions; CTR 0.027% vs band referen...",The broad position band may mix very different...
3,4,content_acbcc847f8996314,Review search intent and snippet,"83,715 impressions; CTR 0.159% vs band referen...",Daily average position hides query-level rank ...
4,5,content_34a70fea29d15f24,Review search intent and snippet,"73,639 impressions; CTR 0.024% vs band referen...",Reference-client medians may not transfer to t...
5,6,content_945d6ff91386c817,Review search intent and snippet,"49,314 impressions; CTR 0.004% vs band referen...",Low CTR may be appropriate for informational q...
6,7,content_1642f339bd6e7c8d,Review search intent and snippet,"52,378 impressions; CTR 0.034% vs band referen...",A few heavily exposed queries can dominate the...
7,8,content_f6116743b00afc2d,Review search intent and snippet,"49,619 impressions; CTR 0.016% vs band referen...",The 15-day window may reflect a temporary even...
8,9,content_b99ea6861864dea5,Review search intent and snippet,"91,474 impressions; CTR 0.202% vs band referen...",An already appropriate title could be harmed b...
9,10,content_36fc1ee501ec072d,Review search intent and snippet,"46,199 impressions; CTR 0.024% vs band referen...",Client measurement coverage does not guarantee...


## 4. Weak picks and limits

The top item is not necessarily the best edit: exposure magnifies a descriptive gap, and a client with many pages can dominate a queue. Inspect the top-ten client count above before assigning all reviewer capacity. Consider per-client caps in a later operational experiment, without changing the frozen study baseline.

**Precision@10 for actionable edits is not measured.** No independent review labels or intervention outcomes are present. A future-window persistence check can be reported as a proxy, never as proof that ten edits would work. The capstone compares observed later-CTR error and directional ranking diagnostics on the same held-out rows, rather than inventing human labels.

## 5. Self-check

Two bucket tables include sample sizes and explicit verdicts. The rule uses only March 1–15 inputs and training-client reference medians. The notebook regenerates the ignored queue CSV and includes ten evidence reviews. Run top to bottom using `work/REPRODUCE.md`.

Source: FlyRank warehouse release v20260703; daily March 2026 partition.